In [31]:
# Code adapted from:
# https://huggingface.co/docs/transformers/en/training (Acc. 22 March 2025)

In [93]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from torch.nn import LayerNorm, functional
from torch.optim import AdamW, lr_scheduler
from transformers.trainer_pt_utils import get_parameter_names
from datasets import Dataset, load_dataset
from pathlib import Path
import pandas as pd
import numpy as np
import evaluate

In [6]:
dataset = load_dataset("json", data_files={"train": ["./out/ot_train.json", "./out/nt_train.json"], "validation": ["./out/ot_test.json", "./out/nt_test.json"]})

In [7]:
print(dataset["train"][0])

{'label': 0, 'text': 'BRCJT BR> >LH> JT CMJ> WJT >R<>'}


In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-cased")

In [9]:
def tokenize_function(data):
    return tokenizer(data["text"], padding="max_length", truncation=True)

In [10]:
tokenized_data = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/5841 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1562 [00:00<?, ? examples/s]

In [11]:
print(tokenized_data)

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 5841
    })
    validation: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1562
    })
})


In [32]:
# Code adapted from:
# https://huggingface.co/docs/transformers/en/tasks/sequence_classification (Acc. 28 March 2025)

In [33]:
id2label = {0: "Jewish", 1: "Christian"}
label2id = {"Jewish": 0, "Christian": 1}

In [34]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-cased", num_labels=2, id2label=id2label, label2id=label2id, torch_dtype="auto")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [36]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [37]:
training_args = TrainingArguments(
    output_dir="peshitta_trainer",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    do_eval=True,
    # seed=SEED
)

In [38]:
# Adapted from: https://github.com/huggingface/transformers/issues/18635#issuecomment-1216860652 (Acc. 27 March 2025)
decay_parameters = get_parameter_names(model, [LayerNorm])
decay_parameters = [name for name in decay_parameters if "bias" not in name]

optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if n in decay_parameters],
            "weight_decay": 0.01,
        },
        {
            "params": [p for n, p in model.named_parameters() if n not in decay_parameters],
            "weight_decay": 0.0,
        },
    ]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=2e-5,
    eps=1e-8
)

# params, lr=0.001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0.01, amsgrad=False, *, maximize=False, foreach=None, capturable=False, differentiable=False, fused=None

In [39]:
scheduler = lr_scheduler.LinearLR(optimizer)

In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [42]:
trainer.save_model(str(Path("./aabert").absolute()))

In [43]:
model = AutoModelForSequenceClassification.from_pretrained(str(Path("./aabert").absolute()))

In [44]:
tokenized_data["validation"][0]["text"]

'WHLJN PT"GM> D>MR MWC> LKLH >JSRJL B<BR> DJWRDNN BMDBR> B<RB> LWQBL SWP BJT PRN WBJT TPL WLBNN WXYRWT WRZHB'

In [49]:
inputs = tokenizer(dataset["validation"][0]["text"], padding="max_length", truncation=True, return_tensors="pt")
outputs = model(**inputs)

In [50]:
outputs.logits

tensor([[-0.0488,  0.1979]], grad_fn=<AddmmBackward0>)

In [59]:
# https://stackoverflow.com/a/60183913 (Acc. 20 March 2025)
prob = functional.softmax(outputs.logits, dim=1)
prob.detach().numpy()

array([[0.43863377, 0.56136626]], dtype=float32)

In [76]:
type(model)
# data collator using DataLoader

transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification

In [ ]:
model.eval() # setting the model to evaluation mode

In [109]:
def predict_all(tuned_model, tokenized_ds: Dataset):
    predictions = np.empty(0, dtype=np.float32)
    for i in range(len(tokenized_ds)):
        outputs = tuned_model(*tokenized_ds)
        predictions = np.append(predictions, outputs)
    return predictions

In [110]:
def calc_proba(model_outputs: np.ndarray) -> np.ndarray:
    result = np.empty(0, dtype=np.float32)
    for classification in model_outputs:
        # Adapted from: https://stackoverflow.com/a/60183913
        prob = functional.softmax(classification.logits, dim=1)
        result = np.append(result, prob.detach().numpy())
    return result

In [111]:
df_tokenized = tokenized_data["validation"].to_pandas()
df_tokenized

,label,text,input_ids,attention_mask
0,0,"WHLJN PT""GM> D>MR MWC> LKLH >JSRJL B<BR> DJWRD...","[101, 160, 9530, 4538, 2249, 22216, 107, 14748...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,0,"MRD> XD<SR JWM""JN MN XWRJB LVWR> DS<JR W<DM> L...","[101, 25827, 2137, 135, 161, 2137, 133, 5833, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,0,"WHW> BCNT >R""B<JN BJRX> DXD<SR BXD BJRX> MLL M...","[101, 160, 3048, 2924, 135, 3823, 15681, 135, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,0,"MN BTR DQVL LSJXWN MLK> D>MWR""J> DJ^TB HW> BXC...","[101, 150, 2249, 27378, 2069, 141, 4880, 2559,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,0,B<BR> DJWRDNN B>R<> DMW>B C^RJ MWC> MPCQ NMWS>...,"[101, 139, 133, 26660, 135, 6027, 15824, 2137,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
1557,1,>T<BJ LH GJR LBH D<M> HN> WMCM<THWN >WQRW W<JN...,"[101, 135, 157, 133, 139, 4538, 149, 3048, 144...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1558,1,TTJD< LKWN HKJL HD> DL<MM> HW >CTDR HN> PWRQN>...,"[101, 157, 1942, 4538, 2137, 133, 149, 2428, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1559,1,WKD HLJN >MR NPQW JHWDJ> WSGJ DRCJN HWW BJNTHWN,"[101, 160, 2428, 2137, 145, 2162, 4538, 2249, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1560,1,W>GR LH PWLWS MN DJLH BJT> WHW> BH TRTJN CNJN ...,"[101, 160, 135, 144, 2069, 149, 3048, 153, 292...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [114]:
df_dropped = df_tokenized.drop(columns=["label", "text"])
res = predict_all(model, df_dropped)

AttributeError: 'str' object has no attribute 'size'

In [113]:
# sentencepiece

In [ ]:
# serious re-implementation for real data